In [1]:

import json, re
import os
import json
import time
from pathlib import Path
from typing import Dict, Any, List, Optional

from autogen import ConversableAgent, UserProxyAgent

# ---- Your existing tool import (same as your notebook)
from Utl import acs_county_data, read_from_csv

In [ ]:
API_KEY=""
API_BASE = "https://api.ai.it.cornell.edu/v1" 


In [ ]:
MODELS = [
    "openai.gpt-5-mini",
    "meta.llama-4-maverick-17b-instruct",
    "google.gemini-2.5-flash",
    "xai.grok-3-mini",
    "llama3.1:8b"
]

CITIES = [
    {"city": "Manhattan",   "state_fips": "36", "county_fips": "061"},
    {"city": "Phoenix",     "state_fips": "04", "county_fips": "013"},
    {"city": "Minneapolis", "state_fips": "27", "county_fips": "053"},
    {"city": "LasVegas",    "state_fips": "32", "county_fips": "003"},
]

OUTDIR = Path("outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUTDIR / "results_summary.jsonl"


In [29]:
# 2) Deterministic hard checks
# =====================================================================
def hard_check_population(candidate: Dict[str, Any], State_fips, county_fips) -> Dict[str, Any]:
    violations = []
    if not isinstance(candidate, dict):
        return {"ok": False, "violations": ["candidate_not_dict"], "metrics": {}}

    if candidate.get("stage") != "population":
        violations.append("stage_must_be_population")

    clusters = candidate.get("clusters", None)
    if not isinstance(clusters, list) or len(clusters) == 0:
        violations.append("clusters_missing_or_empty")
        return {"ok": False, "violations": violations, "metrics": {"violations": len(violations)}}

    ids = []
    total = 0
    for i, c in enumerate(clusters):
        if not isinstance(c, dict):
            violations.append(f"cluster_{i}_not_dict")
            continue
        cid = c.get("cluster_id")
        if not cid or not isinstance(cid, str):
            violations.append(f"cluster_{i}_missing_cluster_id")
        else:
            ids.append(cid)

        pc = c.get("population_count")
        if not isinstance(pc, int) or pc <= 0:
            violations.append(f"cluster_{i}_population_count_invalid")
        else:
            total += pc

        if not c.get("demographic_summary"):
            violations.append(f"cluster_{i}_missing_demographic_summary")
        if not c.get("behavior_archetype"):
            violations.append(f"cluster_{i}_missing_behavior_archetype")

    if len(ids) != len(set(ids)):
        violations.append("cluster_id_not_unique")

    ACS_Total = read_from_csv("ACS/36_109.csv")['Estimate!!Total!!Total population'].iloc[0]
    if total != ACS_Total:
        if total < ACS_Total:
            violations.append(f"total is smaller than ground truth by{abs(total-ACS_Total)}")
        if total > ACS_Total:
            violations.append(f"total is greater than ground truth by{abs(total-ACS_Total)}")

    metrics = {
        "total_population_count": total,
        "num_clusters": len(clusters),
        "violations": len(violations),
    }
    return {"ok": len(violations) == 0, "violations": violations, "metrics": metrics}


def hard_check_activity(candidate: Dict[str, Any], clusters: List[Dict[str, Any]]) -> Dict[str, Any]:
    tol = 1e-3
    violations = []

    if not isinstance(candidate, dict):
        return {"ok": False, "violations": ["candidate_not_dict"], "metrics": {}}

    if candidate.get("stage") != "activity":
        violations.append("stage_must_be_activity")

    ca = candidate.get("clusters_activity", None)
    if not isinstance(ca, list) or len(ca) == 0:
        violations.append("clusters_activity_missing_or_empty")
        return {"ok": False, "violations": violations, "metrics": {"violations": len(violations)}}

    pop_ids = [c["cluster_id"] for c in clusters if isinstance(c, dict) and "cluster_id" in c]
    ca_ids = []
    bad_rows = 0

    required_keys = ["sleep", "work", "meal", "errand", "leisure"]

    for i, item in enumerate(ca):
        if not isinstance(item, dict):
            violations.append(f"clusters_activity_{i}_not_dict")
            continue
        cid = item.get("cluster_id")
        if not cid:
            violations.append(f"clusters_activity_{i}_missing_cluster_id")
            continue
        ca_ids.append(cid)

        hourly = item.get("hourly_activity", None)
        if not isinstance(hourly, list) or len(hourly) != 24:
            violations.append(f"{cid}_hourly_activity_must_have_24_entries")
            continue

        seen_hours = set()
        for hrow in hourly:
            if not isinstance(hrow, dict):
                violations.append(f"{cid}_hour_row_not_dict")
                bad_rows += 1
                continue

            hour = hrow.get("hour")
            if not isinstance(hour, int) or hour < 0 or hour > 23:
                violations.append(f"{cid}_invalid_hour_value")
                bad_rows += 1
                continue

            seen_hours.add(hour)
            s = 0.0
            for k in required_keys:
                v = hrow.get(k)
                if not isinstance(v, (int, float)):
                    violations.append(f"{cid}_hour_{hour}_{k}_not_number")
                    bad_rows += 1
                    continue
                if v < -tol or v > 1.0 + tol:
                    violations.append(f"{cid}_hour_{hour}_{k}_out_of_range")
                    bad_rows += 1
                s += float(v)

            if abs(s - 1.0) > 5e-2:
                violations.append(f"{cid}_hour_{hour}_shares_sum_not_1 (sum={s:.3f})")
                bad_rows += 1

        if seen_hours != set(range(24)):
            missing = sorted(list(set(range(24)) - seen_hours))
            violations.append(f"{cid}_missing_hours_{missing}")

    missing_clusters = sorted(list(set(pop_ids) - set(ca_ids)))
    extra_clusters = sorted(list(set(ca_ids) - set(pop_ids)))
    if missing_clusters:
        violations.append(f"missing_cluster_ids_in_activity: {missing_clusters}")
    if extra_clusters:
        violations.append(f"unknown_cluster_ids_in_activity: {extra_clusters}")

    metrics = {
        "num_population_clusters": len(pop_ids),
        "num_activity_clusters": len(ca_ids),
        "bad_rows": bad_rows,
        "violations": len(violations),
    }
    return {"ok": len(violations) == 0, "violations": violations, "metrics": metrics}

In [68]:
# 3) Hybrid Verification agents
# =====================================================================
def build_llm_config(model_name: str) -> Dict[str, Any]:
    """
    Cornell gateway expects base_url = API_BASE.
    We pass the model string as provided in MODELS list.
    """
    return {
        "config_list": [
            {
                "model": model_name,
                "base_url": API_BASE,
                "api_key": API_KEY,
            }
        ],
        "temperature": 0.2,
    }


def build_agents(llm_config: Dict[str, Any]):
    user_proxy = UserProxyAgent(
        name="Runner",
        human_input_mode="NEVER",
        max_consecutive_auto_reply=0,
        default_auto_reply="TERMINATE",
        #is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"],
        code_execution_config={"use_docker": False},
    )


    population_agent = ConversableAgent(
        name="PopulationGenerator",
        llm_config=llm_config,
        system_message=(
            "You are the Population Agent.\n"
            #"You MUST call fetch_acs_county_data(county_fips, state_fips) first.\n"
            "Then return ONLY valid JSON with schema:\n"
            "{stage:'population', county_fips:str, state_fips:str, clusters:["
            "{cluster_id:str, population_count:int, demographic_summary:str, behavior_archetype:str}"
            "]}.\n"
            "demographic_summary str should be like Age (18-34: X%, 35-54: Y%, 55+: Z%), Income (Low: X%, Middle: Y%, High: Z%), Education (High School: X%, Some College: Y%, Bachelor's+: Z%), Gender (Male: X%, Female: Y%), Employment (Full-time: X%, Part-time: Y%, Unemployed: Z%), Travel Behavior (Public Transit: High, Car: Moderate)\n"
            "Make sure the generated clusters are corresponding to the ACS margins\n"
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
            
        ),
    )

    activity_agent = ConversableAgent(
        name="ActivityGenerator",
        llm_config=llm_config,
        system_message=(
            "You are the Activity Agent.\n"
            "Input will include population clusters.\n"
            "Return ONLY valid JSON with schema:\n"
            "{stage:'activity', clusters_activity:["
            "{cluster_id:str, hourly_activity:["
            "{hour:int, sleep:float, work:float, meal:float, errand:float, leisure:float}"
            " x24]}"
            "]}.\n"
            "Per cluster per hour, shares must sum to 1.\n"
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
        ),
    )

    pop_self_verifier = ConversableAgent(
        name="SelfVerifier",
        llm_config=llm_config,
        system_message=(
            "You are a verifier. You will receive candidate JSON and deterministic check_report.\n"
            #"You MUST call fetch_acs_county_data(county_fips, state_fips) first, the data inside are the ground truth\n"
            "Output ONLY JSON:\n"
            "{should_patch:bool, verification_questions:[str], answers:[str], "
            "patch_plan:[{path:str, op:'replace'|'add'|'remove', value:any, reason:str}]}\n"
            "Use ONLY the candidate/check_report; do not invent external facts."
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
        ),
    )

    act_self_verifier = ConversableAgent(
        name="SelfVerifier",
        llm_config=llm_config,
        system_message=(
            "You are a verifier. You will receive candidate JSON and deterministic check_report.\n"
            "Output ONLY JSON:\n"
            "{should_patch:bool, verification_questions:[str], answers:[str], "
            "patch_plan:[{path:str, op:'replace'|'add'|'remove', value:any, reason:str}]}\n"
            "Use ONLY the candidate/check_report; do not invent external facts."
            "You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }"
        ),
    )

    advocate = ConversableAgent(
        name="Advocate",
        llm_config=llm_config,
        system_message="Defend the candidate and address check_report concerns concisely.",
    )
    skeptic = ConversableAgent(
        name="Skeptic",
        llm_config=llm_config,
        system_message="Stress-test the candidate. Identify concrete inconsistencies or missing info.",
    )
    statistician = ConversableAgent(
        name="Statistician",
        llm_config=llm_config,
        system_message="Only discuss check_report metrics/violations and pass/fail.",
    )
    referee = ConversableAgent(
        name="Referee",
        llm_config=llm_config,
        system_message=(
            "You are the final decision maker.\n"
            "Primary rule: hard constraints dominate (check_report.ok must be true).\n"
            "Output ONLY JSON: {decision:'accept'|'reject', required_fixes:[str]}."
        ),
    )

    # Register tool just like your notebook
    #population_agent.register_for_llm(
    #    name="fetch_acs_county_data",
    #    description="Input: county_fips (str), state_fips (str). Output: ACS summary."
    #)(acs_county_data)
    #pop_self_verifier.register_for_llm(
    #    name="fetch_acs_county_data",
    #    description="Input: county_fips (str), state_fips (str). Output: ACS summary."
    #)(acs_county_data)
    #user_proxy.register_for_execution(name="fetch_acs_county_data")(acs_county_data)

    return user_proxy, population_agent, activity_agent, pop_self_verifier, act_self_verifier, advocate, skeptic, statistician, referee




In [ ]:
# 4) Hybrid stage runner
# =====================================================================
def _parse_json_from_resp(resp) -> Dict[str, Any]:
    return json.loads(resp.chat_history[-1]["content"])




def extract_json_or_raise(text: str) -> dict:
    if not text or not isinstance(text, str):
        raise ValueError("Empty/non-string response")

    s = text.strip()

    # strip markdown fences
    s = re.sub(r"^```(?:json|python)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    # handle print({...})
    m = re.search(r"print\s*\(\s*(\{.*\})\s*\)\s*$", s, flags=re.DOTALL)
    if m:
        s = m.group(1)

    # take first {...} block
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found")

    return json.loads(s[start:end+1])

import json
from typing import Any, Dict, Optional
from autogen import ConversableAgent, UserProxyAgent


def run_stage_hybrid(
    county: str,
    state: str,
    stage: str,
    generator_agent: ConversableAgent,
    prompt: str,
    user_proxy: UserProxyAgent,
    self_verifier: ConversableAgent,
    advocate: ConversableAgent,
    skeptic: ConversableAgent,
    statistician: ConversableAgent,
    referee: ConversableAgent,
    hard_check_fn,
    hard_check_kwargs: Optional[Dict[str, Any]] = None,
    max_rounds: int = 3,
    # NEW: retry controls
    max_json_retries: int = 5,
) -> Dict[str, Any]:
    """
    Hybrid stage runner:
      - Generate candidate JSON (with parse-retry loop)
      - Hard-check
      - Self-check (with parse-retry loop)
      - Optional patch apply (with parse-retry loop)
      - Debate (text)
      - Referee verdict (with parse-retry loop)
    """
    hard_check_kwargs = hard_check_kwargs or {}

    last_check = None
    last_candidate = None
    last_verdict = None

    def _last_assistant_text(resp) -> str:
        """Find the last assistant message content robustly."""
        if hasattr(resp, "chat_history"):
            for m in reversed(resp.chat_history):
                if m.get("role") == "assistant" and isinstance(m.get("content"), str):
                    return m["content"]
            if resp.chat_history:
                c = resp.chat_history[-1].get("content", "")
                return c if isinstance(c, str) else ""
        return ""

    def _chat_until_json(agent, message: str, *, clear: bool = True,) -> Dict[str, Any]:
        resp = user_proxy.initiate_chat(
            recipient=agent,
            message=message,
            clear_history=clear,
        )

        e = None
        last_txt = _last_assistant_text(resp)

        try:
            return _parse_json_from_resp(resp)
        except Exception as ex:
            e = ex

        for attempt in range(1, max_json_retries + 1):
            regen_msg = (
                "Your previous response was NOT valid/extractable JSON.\n"
                "Return ONLY a single valid JSON object.\n"
                "- No code blocks, no markdown, no print(), no variable assignment.\n"
                "- The response must start with '{' and end with '}'.\n\n"
                f"Parsing error: {type(e).__name__}: {e}\n"
                "Previous response (truncated):\n"
                f"{last_txt[:800]}\n"
            )
            resp = user_proxy.send(regen_msg, agent)
            last_txt = _last_assistant_text(resp)
            try:
                return _parse_json_from_resp(resp)
            except Exception as ex2:
                e = ex2

        raise RuntimeError(f"[{stage}] Failed to obtain valid JSON from {agent.name}.")

    
    

    for r in range(1, max_rounds + 1):
        # 1) Generate (JSON retry)
        candidate = _chat_until_json(generator_agent, prompt, clear=True)

        # 2) Hard checks
        check_report = hard_check_fn(candidate, state, county, **hard_check_kwargs)
        last_check, last_candidate = check_report, candidate

        # 3) Self-check (JSON retry)
        vpacket = {"stage": stage, "candidate_json": candidate, "check_report": check_report, "county_fips":county, "state_fips":state}
        vjson = _chat_until_json(self_verifier, json.dumps(vpacket), clear=True)

        # 4) Optional patch (JSON retry)
        if vjson.get("should_patch") and vjson.get("patch_plan"):
            patch_req = {
                "stage": stage,
                "candidate_json": candidate,
                "check_report": check_report,
                "patch_plan": vjson["patch_plan"],
                "instruction": "Apply patch_plan and return revised JSON only.",
            }
            candidate = _chat_until_json(generator_agent, json.dumps(patch_req), clear=False)
            check_report = hard_check_fn(candidate, state, county, **hard_check_kwargs)
            last_check, last_candidate = check_report, candidate

        # 5) Debate (TEXT; no JSON required)
        dpacket = {"stage": stage, "candidate_json": candidate, "check_report": check_report}
        dmsg = json.dumps(dpacket)

        # Use clear_history=True to avoid any lingering state / auto loops
        adv_resp = user_proxy.initiate_chat(recipient=advocate, message=dmsg, clear_history=True)
        skp_resp = user_proxy.initiate_chat(recipient=skeptic, message=dmsg, clear_history=True)
        stat_resp = user_proxy.initiate_chat(recipient=statistician, message=dmsg, clear_history=True)

        adv = _last_assistant_text(adv_resp)
        skp = _last_assistant_text(skp_resp)
        stat = _last_assistant_text(stat_resp)

        # 6) Referee verdict (JSON retry)
        ref_packet = {
            "stage": stage,
            "candidate_json": candidate,
            "check_report": check_report,
            "debate": {"advocate": adv, "skeptic": skp, "statistician": stat},
        }
        verdict = _chat_until_json(referee, json.dumps(ref_packet), clear=True)
        last_verdict = verdict

        if verdict.get("decision") == "accept" and check_report.get("ok", False):
            return {
                "stage": stage,
                "round": r,
                "accepted": candidate,
                "check_report": check_report,
                "verdict": verdict,
            }


    return {
        "stage": stage,
        "round": max_rounds,
        "accepted": None,
        "check_report": last_check,
        "verdict": last_verdict or {"decision": "reject", "required_fixes": ["Max rounds reached."]},
        "last_candidate": last_candidate,
    }


# 5) Prompts
# =====================================================================
def make_population_prompt(city: str, county_fips: str, state_fips: str) -> str:
    acs_text = acs_county_data(county_fips=county_fips, state_fips=state_fips)
    return json.dumps({
        "instruction": "Generate population clusters for the given county/state using ACS. You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }",
        "city": city,
        "county_fips": county_fips,
        "state_fips": state_fips,
        "ACS_marginals": acs_text,
        "requirements": {
            "min_clusters": 4,
            "max_clusters": 8,
            "return_json_only": True,
            "cluster_fields": ["cluster_id", "population_count", "demographic_summary", "behavior_archetype"],
        }
    })

def make_activity_prompt(population_json: Dict[str, Any]) -> str:
    return json.dumps({
        "instruction": "Given population clusters, generate 24h aggregated activity shares per cluster.",
        "population_clusters": population_json["clusters"],
        "activity_types": ["sleep", "work", "meal", "errand", "leisure"],
        "requirements": {
            "hours": list(range(24)),
            "per_hour_sum_to_1": True,
            "return_json_only": True,
        }
    })





In [77]:
# 6) Experiment loop: cities × models
# =====================================================================
def write_json(path: Path, obj: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def append_summary(record: Dict[str, Any]):
    with open(SUMMARY_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def main():
    if not API_KEY:
        raise RuntimeError("OPENAI_API_KEY is empty. Please set it in your environment.")

    for city in CITIES:
        city_name = city["city"]
        state_fips = city["state_fips"]
        county_fips = city["county_fips"]

        for model in MODELS:
            run_id = f"{city_name}__{model}"
            print(f"\n=== Running {run_id} ===")

            llm_config = build_llm_config(model)
            user_proxy, pop_agent, act_agent, pop_self_verifier, act_self_verifier, advocate, skeptic, statistician, referee = build_agents(llm_config)

            t0 = time.time()
            status = "ok"
            err = None

            try:
                # Population
                pop_prompt = make_population_prompt(city_name, county_fips, state_fips)
                pop_res = run_stage_hybrid(
                    stage="population",
                    generator_agent=pop_agent,
                    prompt=pop_prompt,
                    user_proxy=user_proxy,
                    self_verifier=pop_self_verifier,
                    advocate=advocate,
                    skeptic=skeptic,
                    statistician=statistician,
                    referee=referee,
                    hard_check_fn=hard_check_population,
                    max_rounds=3,
                    county=county_fips,
                    state=state_fips,
                )

                if pop_res["accepted"] is None:
                    raise RuntimeError(f"Population rejected: {pop_res['verdict']}")

                # Activity
                act_prompt = make_activity_prompt(pop_res["accepted"])
                act_res = run_stage_hybrid(
                    stage="activity",
                    generator_agent=act_agent,
                    prompt=act_prompt,
                    user_proxy=user_proxy,
                    self_verifier=act_self_verifier,
                    advocate=advocate,
                    skeptic=skeptic,
                    statistician=statistician,
                    referee=referee,
                    hard_check_fn=hard_check_activity,
                    hard_check_kwargs={"clusters": pop_res["accepted"]["clusters"]},
                    max_rounds=3,
                )

                if act_res["accepted"] is None:
                    raise RuntimeError(f"Activity rejected: {act_res['verdict']}")

                # Save accepted outputs
                base = OUTDIR / city_name / model
                write_json(base / "population.json", pop_res["accepted"])
                write_json(base / "activity.json", act_res["accepted"])
                write_json(base / "population_check_report.json", pop_res["check_report"])
                write_json(base / "activity_check_report.json", act_res["check_report"])

                # Summary record
                rec = {
                    "city": city_name,
                    "state_fips": state_fips,
                    "county_fips": county_fips,
                    "model": model,
                    "status": "accepted",
                    "population_rounds": pop_res["round"],
                    "activity_rounds": act_res["round"],
                    "population_metrics": pop_res["check_report"].get("metrics", {}),
                    "activity_metrics": act_res["check_report"].get("metrics", {}),
                    "seconds": round(time.time() - t0, 2),
                }
                append_summary(rec)
                print("Accepted ✔", rec["seconds"], "sec")

            except Exception as e:
                status = "failed"
                err = str(e)
                rec = {
                    "city": city_name,
                    "state_fips": state_fips,
                    "county_fips": county_fips,
                    "model": model,
                    "status": status,
                    "error": err,
                    "seconds": round(time.time() - t0, 2),
                }
                append_summary(rec)
                print("Failed ✘", err)


In [78]:

if __name__ == "__main__":
    main()


=== Running Manhattan__meta.llama-4-maverick-17b-instruct ===
Runner (to PopulationGenerator):

{"instruction": "Generate population clusters for the given county/state using ACS. You must output ONLY valid JSON. No code blocks. No markdown. No extra text. The response must start with { and end with }", "city": "Manhattan", "county_fips": "061", "state_fips": "36", "ACS_marginals": "Loaded cached ACS data from `ACS\\36_061.csv` (1 row(s), 160 column(s))\n\nEstimate!!Total!!Total population,Estimate!!Total!!Total population!!AGE!!Under 5 years,Estimate!!Total!!Total population!!AGE!!5 to 9 years,Estimate!!Total!!Total population!!AGE!!10 to 14 years,Estimate!!Total!!Total population!!AGE!!15 to 19 years,Estimate!!Total!!Total population!!AGE!!20 to 24 years,Estimate!!Total!!Total population!!AGE!!25 to 29 years,Estimate!!Total!!Total population!!AGE!!30 to 34 years,Estimate!!Total!!Total population!!AGE!!35 to 39 years,Estimate!!Total!!Total population!!AGE!!40 to 44 years,Estimate!!To